## Chat with a local Large Language Model

This notebook shows how you can chat with an LLM through an EMISSOR client. The EMISSOR layer will capture the interaction as a scenario for further analysis.
For chatting with a LLama model, we use the [LangChain layer on top of Ollama](https://api.python.langchain.com/en/latest/chat_models/langchain_ollama.chat_models.ChatOllama.html). 

Ollama allows you to pull a model from the web to your local machine and use a ```chat``` function to send instructions to local model to get a response. 
You need to install the ```Ollama``` server locally on you computer. Please follow the instructions on the website. Continue with this notebook after installing the Ollama server.

```
https://github.com/ollama/ollama
```

Instead of the Ollama ```chat``` function, we will create a client through ChatOllama so that we can set parameters for the behaviour.

There are various versions of models. We are going to pull the smallest Llama3.2 model that already gives reasonable performance but only works for text input.
To be able to access the model through Ollama, we need to pull it from the terminal in the same virtual environment:

```
ollama pull llama3.2:1b
```

You only need to do this once.

You can use the following command to check what models are downloaded and available.


In [9]:
!ollama list

NAME               ID              SIZE      MODIFIED      
llama3.2:1b        baf6a787fdff    1.3 GB    3 minutes ago    
qwen3:0.6b         7df6b6e09427    522 MB    2 months ago     
qwen3:latest       500a1f067a9f    5.2 GB    2 months ago     
qwen3:1.7b         8f68893c685c    1.4 GB    2 months ago     
qwen2.5:latest     845dbda0ea48    4.7 GB    7 months ago     
llama3.2:latest    a80c4f17acd5    2.0 GB    11 months ago    


### Loading an LLM in ChatOllama

In [21]:
from langchain_ollama import ChatOllama
llm_model = "llama3.2:1b"
#llm_model = "qwen3:1.7b"
#llm_model= "qwen3:0.6b"

llm = ChatOllama(
    model = llm_model,
    temperature = 0.8,
    num_predict = 256,
    # other params ...
)

In [23]:
instruct = { 'role': 'system', 'content': "You are a docter and you will receive questions from patients. Be brief and no more than one sentence and 15 words or less."}

### Creating an EMISSOR client

The EMISSOR client is used to create a scenario folder in the emissor_path each time you create a client instance. Any interaction will be represented as a sequence of signals by a source. At the end of the interaction the actions of the interlocutors are saved in the scenario folder.

In [17]:
from leolani_client import LeolaniChatClient
emissor_path = "./emissor"
HUMAN="Piek"
AGENT="LLM"
leolaniClient = LeolaniChatClient(emissor_path=emissor_path, agent=AGENT, human=HUMAN)

### Interaction loop

The next interaction loop shows how we send the input to the LLM by calling the ```invoke``` function on the history.
The history is the list of user input and LLM responses, structured as a dictionary with the keys ```role``` and ```content```.
For the human input the role is ```user``` and for the LLM response the role is ```system```.
The first element in the history is the instruction.

In addition to the prompts and the history, we also add each utterance to the ```leolaniClient``` which will store it as an EMISSOR signal.
After terminating the while loop, we save the interaction through the ```leolaniClient``` to disk in the EMISSOR scenario folder.

In [18]:
### We keep track of the history of the conversations.
history = []
history.append(instruct)
print(history)
### First prompt
response = llm.invoke(history)
utterance = response.content
print(AGENT + ": " + utterance)
leolaniClient._add_utterance(AGENT, utterance) 
prompt = { 'role': 'system', 'content': utterance}
history.append(prompt)

utterance = input("\n")
print(HUMAN + ": " + utterance)
leolaniClient._add_utterance(HUMAN, utterance)
prompt = { 'role': 'user', 'content': utterance}
history.append(prompt)

while not (utterance.lower() == "stop" or utterance.lower() == "bye"):
    # Create the response from the system and store this as a new signal
    response = llm.invoke(history)
    utterance = response.content
    print(AGENT + ": " + utterance)
    leolaniClient._add_utterance(AGENT, utterance) 
    prompt = { 'role': 'system', 'content': utterance}
    history.append(prompt)

    utterance = input("\n")
    print(HUMAN + ": " + utterance)
    leolaniClient._add_utterance(HUMAN, utterance)
    prompt = { 'role': 'user', 'content': utterance}
    history.append(prompt)

##### After completion, we save the scenario in the defined emissor folder.
leolaniClient._save_scenario() 

[{'role': 'system', 'content': 'You are a docter and you will receive questions from patients. Be brief and no more than two sentences.'}]
LLM: <think>
Okay, the user is a patient who wants to ask me questions, but I need to respond in two sentences, brief, and keep it simple. Let me think about the most important aspects first. They might be asking about medical information, treatment plans, or health conditions. I should keep it straightforward and avoid unnecessary details. Make sure to use short sentences and keep it polite and professional.
</think>

1. Please provide any specific questions or concerns you have.  
2. I'll help you understand your health situation in two sentences.



 bye


Piek: bye


Note that running the interaction loop again continues the conversation in the same EMISSOR scenario. Only when you create a new instance of the ```LeolaniChatClient``` you will also create a new scenario.

## End of notebook